# Variational method: particle in a harmonic oscillator

- Wavefunction: $\psi(x, \alpha, \beta) = e^{-\alpha x² + \beta x⁴}$ with the variational parameters $\alpha$ and $\beta$
- Hamiltionian: $\hat{H} = - \frac{\hbar}{2m} \nabla² + \frac{1}{2} mw²x²$

We should find $\alpha = 1/2, \beta = 0$.

(Not the most optimal way to do it, but closer to the physical and mathematical procedure that you would do by hand - except the parameter space exploration)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, diff, exp, lambdify
from scipy.optimize import minimize

In [2]:
x = symbols('x')
alpha_symbolic, beta_symbolic = symbols('alpha beta')

def psi(x, alpha, beta):
    '''
    Variational wavefunction for the harmonic oscillator with parameters alpha and beta.
    '''
    return exp(-alpha * x**2 + beta * x**4)

def H_sym(func):
    '''
    Symbolic Hamiltonian operator for the harmonic oscillator applied to the wavefunction.
    '''
    # Kinetic energy term
    T = -0.5 * diff(func, x, 2)
    # Potential energy term
    V = 0.5 * x**2 * func

    return T + V

def H(x_vals, alpha_val, beta_val, func):
    '''
    Hamiltonian operator for the harmonic oscillator applied to the wavefunction.
    '''
    H_symbolic = H_sym(func)

    # Symbolic expression into numerical function
    H_num = lambdify((x, alpha_symbolic, beta_symbolic), H_symbolic, 'numpy')

    return H_num(x_vals, alpha_val, beta_val), H_symbolic

# Compute the expectation value of the Hamiltonian
def energy(x_vals, alpha_val, beta_val):
    '''
    Compute the expectation value of the Hamiltonian for given alpha and beta over x_vals.
    '''
    # Symbolic wavefunction
    psi_symbolic = psi(x, alpha_symbolic, beta_symbolic)

    # Numerical wavefunction
    psi_num = lambdify((x, alpha_symbolic, beta_symbolic), psi_symbolic, 'numpy')
    psi_vals = psi_num(x_vals, alpha_val, beta_val)

    # Apply the Hamiltonian to the wavefunction
    H_vals = H(x_vals, alpha_val, beta_val, psi_symbolic)[0]

    E = np.trapezoid(np.conjugate(psi_vals) * H_vals, x_vals) / (np.trapezoid(np.conjugate(psi_vals) * psi_vals, x_vals))

    return E

In [5]:
# space of parameters to explore
x_vals = np.linspace(-3, 3, 500)
alpha_vals = np.linspace(0.01, 2, 100)
beta_vals = np.linspace(-0.5, 0, 100)

energies = np.zeros((len(alpha_vals), len(beta_vals)))

for i, alpha in enumerate(alpha_vals):
    for j, beta in enumerate(beta_vals):
        energies[i, j] = energy(x_vals, alpha, beta)

# lowest energy and corresponding parameters
min_idx = np.unravel_index(np.argmin(energies), energies.shape)
print("Lowest energy:", energies[min_idx])
print("Optimal alpha:", alpha_vals[min_idx[0]])
print("Optimal beta:", beta_vals[min_idx[1]])

Lowest energy: 0.5000546463713348
Optimal alpha: 0.49242424242424243
Optimal beta: 0.0


In [6]:
def optimize_energy(x_vals):
    '''
    Optimize the energy with respect to alpha and beta for given x_vals.
    '''
    # Minimize the numerical expectation value of the Hamiltonian
    res = minimize(
        lambda p: energy(x_vals, p[0], p[1]),
        x0=[0.1, -0.01],   # alpha, beta initial
        method='Nelder-Mead'
    )

    alpha_opt, beta_opt = res.x
    E_opt = res.fun

    return alpha_opt, beta_opt, E_opt

# energy with the optimized alpha and beta from derivative minimization
alpha_opt, beta_opt, E_opt = optimize_energy(x_vals)
print(f"Lowest energy: {E_opt}")
print(f"Optimized alpha: {alpha_opt}")
print(f"Optimized beta: {beta_opt}")

Lowest energy: 0.4999980536912736
Optimized alpha: 0.5015105401121581
Optimized beta: 0.000587765921106624
